Load in dataframes from previous notebook

In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
INTERIM = ROOT / "data" / "interim"

metros = pd.read_parquet(INTERIM / "fmr_metros.parquet")
ds = pd.read_parquet(INTERIM / "oews_ds.parquet")

In [2]:
display(metros.head())
display(ds.head())

,OBJECTID,FMR_CODE,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR,Shape__Area,Shape__Length
0,1,METRO10180M10180,"Abilene, TX MSA",1010,1020,1276,1653,1987,1.005900e+10,4.614773e+05
1,3,METRO10420M10420,"Akron, OH MSA",904,985,1268,1547,1681,4.225362e+09,2.914861e+05
2,4,METRO10500M10500,"Albany, GA MSA",966,1006,1129,1534,1858,5.767117e+09,4.128615e+05
3,6,METRO10580M10580,"Albany-Schenectady-Troy, NY MSA",1217,1417,1702,2041,2253,1.384950e+10,6.475938e+05
4,7,METRO10740M10740,"Albuquerque, NM MSA",1009,1185,1464,2036,2399,3.607428e+10,1.106411e+06


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,15-2051,Data Scientists,...,40.19,60.04,71.70,38490.0,52000.0,83600.0,124890.0,149140.0,None,None
1,10420,"Akron, OH",4,OH,0,Cross-industry,cross-industry,1235,15-2051,Data Scientists,...,47.97,62.99,66.68,66310.0,79370.0,99780.0,131010.0,138690.0,None,None
2,10500,"Albany, GA",4,GA,0,Cross-industry,cross-industry,1235,15-2051,Data Scientists,...,46.80,58.10,68.97,54440.0,72030.0,97350.0,120840.0,143450.0,None,None
3,10580,"Albany-Schenectady-Troy, NY",4,NY,0,Cross-industry,cross-industry,1235,15-2051,Data Scientists,...,47.57,62.04,77.14,64450.0,77330.0,98940.0,129030.0,160460.0,None,None
4,10740,"Albuquerque, NM",4,NM,0,Cross-industry,cross-industry,1235,15-2051,Data Scientists,...,47.48,56.42,63.82,58260.0,69700.0,98750.0,117350.0,132750.0,None,None


Split "FMR_CODE" into two columns. "AREA" is the column that I will be joining the datasets on. 

In [8]:
metros["FMR_CODE"] = metros["FMR_CODE"].astype(str).str.strip()
metros["AREA"] = metros["FMR_CODE"].str[5:10]
metros["SUBDIV"] = metros["FMR_CODE"].str[10:]

metros[["FMR_CODE", "AREA", "SUBDIV"]].head()

,FMR_CODE,AREA,SUBDIV
0,METRO10180M10180,10180,M10180
1,METRO10420M10420,10420,M10420
2,METRO10500M10500,10500,M10500
3,METRO10580M10580,10580,M10580
4,METRO10740M10740,10740,M10740


Next, I will remove the variables that are unecessary, such as "OCC_CODE" in the oews dataframe. This code is constant accross the data. 

In [24]:
for col in ds.columns:
    if ds[col].nunique() <= 1:
        print(col)

AREA_TYPE
NAICS
NAICS_TITLE
I_GROUP
OWN_CODE
OCC_CODE
OCC_TITLE
O_GROUP
PCT_TOTAL
PCT_RPT
ANNUAL
HOURLY


In [25]:
metros_clean = metros.drop(columns=["OBJECTID", "FMR_CODE", ])
ds_clean = ds.drop(columns=["AREA_TYPE", "NAICS", "NAICS_TITLE", "I_GROUP", "OWN_CODE", "OCC_CODE", "OCC_TITLE", "O_GROUP", "PCT_TOTAL", "PCT_RPT", "ANNUAL", "HOURLY"])

In [27]:
display(metros_clean.head())
display(ds_clean.head())

,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR,Shape__Area,Shape__Length,AREA,SUBDIV
0,"Abilene, TX MSA",1010,1020,1276,1653,1987,1.005900e+10,4.614773e+05,10180,M10180
1,"Akron, OH MSA",904,985,1268,1547,1681,4.225362e+09,2.914861e+05,10420,M10420
2,"Albany, GA MSA",966,1006,1129,1534,1858,5.767117e+09,4.128615e+05,10500,M10500
3,"Albany-Schenectady-Troy, NY MSA",1217,1417,1702,2041,2253,1.384950e+10,6.475938e+05,10580,M10580
4,"Albuquerque, NM MSA",1009,1185,1464,2036,2399,3.607428e+10,1.106411e+06,10740,M10740


,AREA,AREA_TITLE,PRIM_STATE,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90
0,10180,"Abilene, TX",TX,40.0,10.4,0.574,0.34,43.85,91210.0,5.2,18.50,25.00,40.19,60.04,71.70,38490.0,52000.0,83600.0,124890.0,149140.0
1,10420,"Akron, OH",OH,270.0,8.1,0.845,0.50,50.40,104840.0,1.6,31.88,38.16,47.97,62.99,66.68,66310.0,79370.0,99780.0,131010.0,138690.0
2,10500,"Albany, GA",GA,50.0,9.5,0.841,0.50,47.30,98380.0,3.7,26.17,34.63,46.80,58.10,68.97,54440.0,72030.0,97350.0,120840.0,143450.0
3,10580,"Albany-Schenectady-Troy, NY",NY,830.0,5.3,1.856,1.10,51.54,107210.0,1.7,30.99,37.18,47.57,62.04,77.14,64450.0,77330.0,98940.0,129030.0,160460.0
4,10740,"Albuquerque, NM",NM,360.0,13.0,0.887,0.53,46.71,97160.0,3.0,28.01,33.51,47.48,56.42,63.82,58260.0,69700.0,98750.0,117350.0,132750.0
